# Jour 3 · Projet final — comparer et déployer


## Objectifs

- comparer une méthode statistique, Isolation Forest et un autoencoder
- produire une table d'alertes priorisées
- dessiner la place du détecteur dans une architecture IoT

Vous êtes responsable du prototype de surveillance de `hvac_01`. Comparez trois détecteurs sur les quatorze derniers jours. Un bon résultat n'est pas seulement un score : il doit produire des alertes compréhensibles et une décision exploitable.

![Tableau comparant seuil statistique, Isolation Forest et autoencoder](../assets/jour_03/04_comparer_les_methodes.png)

*Les trois méthodes n'offrent pas le même compromis entre explicabilité, complexité et capacité à représenter plusieurs capteurs.*

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_labeled.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.set_index("timestamp").sort_index()
features = ["temperature_c", "humidity_pct", "power_kw", "pressure_bar", "vibration_mm_s"]
train_end = df.index.min() + pd.Timedelta(days=28)
evaluation_start = df.index.max() - pd.Timedelta(days=14)

### À vous de jouer — 1. détecteur statistique

Calculez le Z-score mobile de la vibration avec deux jours de passé, décalé d'un pas. Signalez les scores supérieurs à 4 dans `stat_flag`.

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — 2. Isolation Forest

Entraînez Isolation Forest sur les observations normales des 28 premiers jours, puis créez `iforest_flag` et `iforest_score`.

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — 3. autoencoder

Standardisez les données, entraînez l'autoencoder sur le même passé normal et créez `autoencoder_flag` avec le quantile 0,99.

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — 4. comparer honnêtement

Sur les quatorze derniers jours, calculez pour chaque méthode le nombre d'alertes, la précision, le rappel et le F1.

In [ ]:
# Écrivez votre code ici.
pass

![Trois détecteurs votant pour créer une file d'intervention priorisée](../assets/jour_03/04_vote_et_priorisation.png)

*Demander l'accord d'au moins deux méthodes peut réduire les alertes isolées avant l'ajout de la sévérité et de l'action recommandée.*

### À vous de jouer — 5. créer une file d'intervention

Créez une alerte prioritaire quand au moins deux méthodes sont d'accord. Ajoutez une sévérité et une action recommandée, puis affichez les dix plus récentes.

In [ ]:
# Écrivez votre code ici.
pass

![Architecture de production avec données, variables, artefacts versionnés, score, alertes et technicien](../assets/jour_03/04_architecture_production.png)

*Le scaler, le modèle et le seuil font partie du système à versionner ; l'historique des alertes et le retour terrain ferment la boucle.*

## Discussion finale d'architecture

Dessinez le trajet complet : capteur, protocole, broker, stockage, nettoyage, calcul des variables, modèle, seuil, service d'alertes, tableau de bord et retour du technicien.

Pour chaque bloc, précisez : que se passe-t-il si les données cessent d'arriver ? Où versionner le modèle et le scaler ? Comment ajuster le seuil ? Qui confirme qu'une alerte correspondait réellement à un défaut ?

**Conclusion :** la meilleure méthode n'est pas forcément la plus complexe. Elle est celle dont les erreurs, le coût, l'exploitation et la maintenance sont acceptables dans le système réel.